In [ ]:
import os
import sys
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf

import quest.utils.utils as qutils

# -----------------------------
# IMPORTANT: set your QueST repo path
# -----------------------------
QUEST_REPO = "/NHNHOME/WORKSPACE/0226010443_A/seunghyo/real_robot/QueST"
CONFIG_DIR = os.path.join(QUEST_REPO, "config")
AUTOENCODER_ROOT = os.path.join(
    QUEST_REPO,
    "experiments/real_robot/REAL_ROBOT_MULTI",
)

if QUEST_REPO not in sys.path:
    sys.path.append(QUEST_REPO)

OmegaConf.register_new_resolver("eval", eval, replace=True)

device = "cuda:5"
data_prefix = "/NHNHOME/WORKSPACE/0226010443_A/seunghyo/real_robot/demos"


def _infer_extra_overrides(algo_name, exp_name, variant_name):
    extra = []
    if algo_name == "quest_ft_adaln":
        mode = variant_name.removeprefix("block_32_ds_4_ft_")
        if mode not in {"max", "avg", "avg_max", "conv"}:
            raise ValueError(f"Cannot infer ft_downsample_mode from variant_name={variant_name}")
        extra.append(f"algo.ft_downsample_mode={mode}")

        if exp_name.endswith("_100hz"):
            extra.append("algo.dataset.ft_config.ft_source=force_history")
        elif exp_name.endswith("_10hz"):
            extra.append("algo.dataset.ft_config.ft_source=state")
    return extra


def _cfg_from_stage0_dir(stage0_dir):
    stage0_dir = Path(stage0_dir)
    seed = stage0_dir.parent.name
    variant_name = stage0_dir.parent.parent.name
    exp_name = stage0_dir.parent.parent.parent.name
    algo_name = stage0_dir.parent.parent.parent.parent.name

    overrides = [
        "task=real_robot_state",
        f"algo={algo_name}",
        f"exp_name={exp_name}",
        f"variant_name={variant_name}",
        "training.use_tqdm=false",
        "training.save_all_checkpoints=true",
        "training.use_amp=false",
        "train_dataloader.persistent_workers=true",
        "train_dataloader.num_workers=6",
        "train_dataloader.multiprocessing_context=fork",
        "make_unique_experiment_dir=false",
        "algo.skill_block_size=32",
        "algo.downsample_factor=4",
        f"seed={seed}",
        f"data_prefix={data_prefix}",
        f"device={device}",
    ]
    overrides.extend(_infer_extra_overrides(algo_name, exp_name, variant_name))

    with initialize_config_dir(config_dir=CONFIG_DIR, version_base=None):
        cfg = compose(config_name="train_autoencoder.yaml", overrides=overrides)

    # Keep compatibility with older checkpoints/notebooks that expected this explicit FSQ level.
    cfg["algo"]["policy"]["autoencoder"]["fsq_level"] = [8, 5, 5, 5]
    return cfg, algo_name, exp_name, variant_name, seed


def load_autoencoder_from_stage0(stage0_dir):
    cfg, algo_name, exp_name, variant_name, seed = _cfg_from_stage0_dir(stage0_dir)
    model = instantiate(cfg.algo.policy, shape_meta=cfg.task.shape_meta)
    model = model.to(device)
    model.eval()

    checkpoint_path = qutils.get_latest_checkpoint(str(stage0_dir))
    state_dict = qutils.load_state(checkpoint_path)
    qutils.soft_load_state_dict(model, state_dict["model"])
    model.eval()
    model.autoencoder.eval()

    rel_stage0_dir = os.path.relpath(stage0_dir, QUEST_REPO)
    key = f"{algo_name}/{exp_name}/{variant_name}/{seed}"
    return key, {
        "model": model,
        "autoencoder": model.autoencoder,
        "cfg": cfg,
        "checkpoint_path": checkpoint_path,
        "stage0_dir": str(stage0_dir),
        "relative_stage0_dir": rel_stage0_dir,
        "algo_name": algo_name,
        "exp_name": exp_name,
        "variant_name": variant_name,
        "seed": seed,
    }


stage0_dirs = sorted(Path(AUTOENCODER_ROOT).glob("*/*/*/*/stage_0"))
if not stage0_dirs:
    raise FileNotFoundError(f"No stage_0 autoencoder directories found under {AUTOENCODER_ROOT}")

autoencoder_models = {}
for stage0_dir in stage0_dirs:
    key, item = load_autoencoder_from_stage0(stage0_dir)
    autoencoder_models[key] = item
    print(f"[{len(autoencoder_models):02d}] {key}")
    print(f"     checkpoint: {item['checkpoint_path']}")
    print(f"     autoencoder: {type(item['autoencoder']).__name__}")

# Backward-compatible aliases for existing visualization cells.
def set_active_autoencoder(key=None):
    global active_key, active_item, model, autoencoder, cfg
    if key is None:
        key = next(iter(autoencoder_models))
    active_key = key
    active_item = autoencoder_models[key]
    model = active_item["model"]
    autoencoder = active_item["autoencoder"]
    cfg = active_item["cfg"]
    print(f"Active autoencoder: {active_key}")
    return active_item

set_active_autoencoder()
print(f"Loaded {len(autoencoder_models)} autoencoders.")
print("Available keys:")
for key in autoencoder_models:
    print(" -", key)


: 

In [ ]:
import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


def get_force_torque_mag_from_episode(ep):
    """
    return:
        force_mag:  (T,)
        torque_mag: (T,)
    """
    force_mags = []
    torque_mags = []

    for obs in ep["observations"]:
        state = _extract_state_from_obs(obs)

        force = state[RIGHT_FORCE_SLICE]
        torque = state[RIGHT_TORQUE_SLICE]

        force_mags.append(float(np.linalg.norm(force)))
        torque_mags.append(float(np.linalg.norm(torque)))

    return (
        np.asarray(force_mags, dtype=np.float32),
        np.asarray(torque_mags, dtype=np.float32),
    )


def get_skill_force_torque_mag_separate(
    ep,
    valid_T,
    downsample_factor=DOWNSAMPLE_FACTOR,
):
    """
    각 skill token에 대응하는 4-step force/torque magnitude 평균
    return:
        skill_force_mag:  (H,)
        skill_torque_mag: (H,)
    """
    force_mag, torque_mag = get_force_torque_mag_from_episode(ep)

    force_mag = force_mag[:valid_T]
    torque_mag = torque_mag[:valid_T]

    H = valid_T // downsample_factor

    skill_force_mag = []
    skill_torque_mag = []

    for h in range(H):
        s = h * downsample_factor
        e = (h + 1) * downsample_factor

        skill_force_mag.append(float(np.mean(force_mag[s:e])))
        skill_torque_mag.append(float(np.mean(torque_mag[s:e])))

    return (
        np.asarray(skill_force_mag, dtype=np.float32),
        np.asarray(skill_torque_mag, dtype=np.float32),
    )


def count_unique_skills_by_percentile_bins(skill_indices, values, num_bins=10):
    """
    values 기준 percentile bin별 unique skill index 수 계산
    """
    skill_indices = np.asarray(skill_indices)
    values = np.asarray(values, dtype=np.float32)

    assert len(skill_indices) == len(values)

    percentiles = np.linspace(0, 100, num_bins + 1)
    edges = np.percentile(values, percentiles)

    unique_counts = []
    bin_labels = []

    for i in range(num_bins):
        lo = edges[i]
        hi = edges[i + 1]

        if i == num_bins - 1:
            mask = (values >= lo) & (values <= hi)
        else:
            mask = (values >= lo) & (values < hi)

        unique_count = len(set(skill_indices[mask].tolist()))
        unique_counts.append(unique_count)

        bin_labels.append(f"{i*10}-{(i+1)*10}%")

    return unique_counts, bin_labels, edges

def count_unique_skills_by_value_bins(skill_indices, values, num_bins=10):
    skill_indices = np.asarray(skill_indices)
    values = np.asarray(values, dtype=np.float32)

    edges = np.linspace(values.min(), values.max(), num_bins + 1)

    unique_counts = []
    token_ratios = []
    bin_labels = []

    total_tokens = len(values)

    for i in range(num_bins):
        lo = edges[i]
        hi = edges[i + 1]

        if i == num_bins - 1:
            mask = (values >= lo) & (values <= hi)
        else:
            mask = (values >= lo) & (values < hi)

        unique_counts.append(len(set(skill_indices[mask].tolist())))
        token_ratios.append(float(np.sum(mask)) / total_tokens * 100.0)
        bin_labels.append(f"{lo:.2f}-{hi:.2f}")

    return unique_counts, token_ratios, bin_labels, edges

def plot_unique_skill_hist_force_torque_for_task(
    model,
    task_path: str,
    device: str,
    num_bins: int = 10,
):
    """
    task_path 아래의 모든 pkl을 읽고,
    force percentile bin / torque percentile bin별 unique skill index 수를 histogram으로 그림.

    Example:
        plot_unique_skill_hist_force_torque_for_task(
            model=model,
            task_path="/home/seunghyo/real_robot/demos/multi_usb",
            device=device,
        )
    """

    pkl_paths = sorted(
        [
            p for p in glob.glob(os.path.join(task_path, "**", "*.pkl"), recursive=True)
            if "demos" in os.path.basename(p)
        ]
    )

    print(f"Found pkl files: {len(pkl_paths)}")

    all_skill_indices = []
    all_skill_force_mag = []
    all_skill_torque_mag = []

    total_eps = 0
    valid_eps = 0

    for pkl_path in pkl_paths:
        episodes = load_episodes_from_pkl(pkl_path)
        total_eps += len(episodes)

        for ep in episodes:
            trimmed = trim_zero_action_episode(ep)
            if trimmed is None:
                continue

            action_np = get_right_actions_from_episode(trimmed)
            T = len(action_np)

            valid_T = (T // DOWNSAMPLE_FACTOR) * DOWNSAMPLE_FACTOR
            if valid_T < DOWNSAMPLE_FACTOR:
                continue

            action_np = action_np[:valid_T]

            _, indices, _ = extract_skill_codes_and_indices(
                model,
                action_np,
                device,
            )

            force_mag, torque_mag = get_skill_force_torque_mag_separate(
                trimmed,
                valid_T=valid_T,
                downsample_factor=DOWNSAMPLE_FACTOR,
            )

            H = min(len(indices), len(force_mag), len(torque_mag))

            all_skill_indices.append(np.asarray(indices[:H]))
            all_skill_force_mag.append(force_mag[:H])
            all_skill_torque_mag.append(torque_mag[:H])

            valid_eps += 1

    if len(all_skill_indices) == 0:
        print("No valid skill data found.")
        return

    all_skill_indices = np.concatenate(all_skill_indices, axis=0)
    all_skill_force_mag = np.concatenate(all_skill_force_mag, axis=0)
    all_skill_torque_mag = np.concatenate(all_skill_torque_mag, axis=0)

    print(f"Total episodes: {total_eps}")
    print(f"Valid episodes: {valid_eps}")
    print(f"Total skill tokens: {len(all_skill_indices)}")
    print(f"Total unique skills: {len(set(all_skill_indices.tolist()))}")

    # force_counts, bin_labels, force_edges = count_unique_skills_by_percentile_bins(
    #     all_skill_indices,
    #     all_skill_force_mag,
    #     num_bins=num_bins,
    # )

    # torque_counts, _, torque_edges = count_unique_skills_by_percentile_bins(
    #     all_skill_indices,
    #     all_skill_torque_mag,
    #     num_bins=num_bins,
    # )

    force_counts, force_token_ratios, force_bin_labels, force_edges = (
        count_unique_skills_by_value_bins(
            all_skill_indices,
            all_skill_force_mag,
            num_bins=num_bins,
        )
    )

    torque_counts, torque_token_ratios, torque_bin_labels, torque_edges = (
        count_unique_skills_by_value_bins(
            all_skill_indices,
            all_skill_torque_mag,
            num_bins=num_bins,
        )
    )

    # bin별 token 비율 계산
    def token_ratio_by_edges(values, edges, num_bins=10):
        values = np.asarray(values, dtype=np.float32)
        total = len(values)
        ratios = []

        for i in range(num_bins):
            lo = edges[i]
            hi = edges[i + 1]

            if i == num_bins - 1:
                mask = (values >= lo) & (values <= hi)
            else:
                mask = (values >= lo) & (values < hi)

            ratios.append(float(np.sum(mask)) / total * 100.0)

        return ratios

    # force_token_ratios = token_ratio_by_edges(
    #     all_skill_force_mag,
    #     force_edges,
    #     num_bins=num_bins,
    # )

    # torque_token_ratios = token_ratio_by_edges(
    #     all_skill_torque_mag,
    #     torque_edges,
    #     num_bins=num_bins,
    # )

    task_name = Path(task_path).name

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

    # x = np.arange(len(bin_labels))
    x = np.arange(len(force_bin_labels))
    width = 0.4

    # -----------------------------
    # Force subplot
    # -----------------------------
    ax1 = axes[0]
    ax2 = ax1.twinx()

    ax1.bar(
        x - width / 2,
        force_counts,
        width=width,
        label="Unique skill count",
    )
    ax2.bar(
        x + width / 2,
        force_token_ratios,
        width=width,
        alpha=0.45,
        label="Step ratio (%)",
    )

    ax1.set_title(f"{task_name} - Force Magnitude Percentile")
    ax1.set_ylabel("Unique skill count")
    ax2.set_ylabel("Step ratio (%)")
    ax1.grid(axis="y", alpha=0.3)
    ax1.set_xticks(x)
    ax1.set_xticklabels(force_bin_labels)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

    # -----------------------------
    # Torque subplot
    # -----------------------------
    ax1 = axes[1]
    ax2 = ax1.twinx()

    ax1.bar(
        x - width / 2,
        torque_counts,
        width=width,
        label="Unique skill count",
    )
    ax2.bar(
        x + width / 2,
        torque_token_ratios,
        width=width,
        alpha=0.45,
        label="Step ratio (%)",
    )

    ax1.set_title(f"{task_name} - Torque Magnitude Percentile")
    ax1.set_ylabel("Unique skill count")
    ax2.set_ylabel("Step ratio (%)")
    ax1.set_xlabel("Magnitude value bin")
    ax1.set_xticks(x)
    # ax1.set_xticklabels(bin_labels, rotation=45)
    ax1.set_xticklabels(torque_bin_labels)
    ax1.grid(axis="y", alpha=0.3)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

    plt.tight_layout()
    plt.show()

# -----------------------------------------------------------------------------
# Updated binning/plotting helpers.
# bin_mode="value" uses equal-width force/torque magnitude bins.
# bin_mode="percentile" uses equal-percentile bins.
# -----------------------------------------------------------------------------

def count_unique_skills_by_bins(skill_indices, values, num_bins=10, bin_mode="value"):
    skill_indices = np.asarray(skill_indices)
    values = np.asarray(values, dtype=np.float32)

    assert len(skill_indices) == len(values)
    if len(values) == 0:
        raise ValueError("values is empty")

    if bin_mode == "value":
        edges = np.linspace(values.min(), values.max(), num_bins + 1)
        bin_labels = [f"{edges[i]:.2f}-{edges[i + 1]:.2f}" for i in range(num_bins)]
        x_label = "Magnitude value bin"
    elif bin_mode == "percentile":
        percentiles = np.linspace(0, 100, num_bins + 1)
        edges = np.percentile(values, percentiles)
        bin_labels = [f"{int(percentiles[i])}-{int(percentiles[i + 1])}%" for i in range(num_bins)]
        x_label = "Magnitude percentile bin"
    else:
        raise ValueError(f"Unsupported bin_mode={bin_mode}. Expected 'value' or 'percentile'.")

    unique_counts = []
    token_ratios = []
    total_tokens = len(values)

    for i in range(num_bins):
        lo = edges[i]
        hi = edges[i + 1]

        if i == num_bins - 1:
            mask = (values >= lo) & (values <= hi)
        else:
            mask = (values >= lo) & (values < hi)

        unique_counts.append(len(set(skill_indices[mask].tolist())))
        token_ratios.append(float(np.sum(mask)) / total_tokens * 100.0)

    return unique_counts, token_ratios, bin_labels, edges, x_label


def plot_unique_skill_hist_force_torque_for_task(
    model,
    task_path: str,
    device: str,
    num_bins: int = 10,
    bin_mode: str = "value",
    model_label: str | None = None,
    output_dir: str | os.PathLike | None = None,
):
    """
    Plot unique skill counts by force/torque magnitude bins for one task.

    Args:
        bin_mode: "value" for equal-width magnitude bins, or "percentile" for percentile bins.
        output_dir: If provided, saves the figure as a PNG in this directory.
    """
    pkl_paths = sorted(
        [
            p for p in glob.glob(os.path.join(task_path, "**", "*.pkl"), recursive=True)
            if "demos" in os.path.basename(p)
        ]
    )

    print(f"Found pkl files: {len(pkl_paths)}")

    all_skill_indices = []
    all_skill_force_mag = []
    all_skill_torque_mag = []

    total_eps = 0
    valid_eps = 0

    for pkl_path in pkl_paths:
        episodes = load_episodes_from_pkl(pkl_path)
        total_eps += len(episodes)

        for ep in episodes:
            trimmed = trim_zero_action_episode(ep)
            if trimmed is None:
                continue

            action_np = get_right_actions_from_episode(trimmed)
            T = len(action_np)

            valid_T = (T // DOWNSAMPLE_FACTOR) * DOWNSAMPLE_FACTOR
            if valid_T < DOWNSAMPLE_FACTOR:
                continue

            action_np = action_np[:valid_T]

            _, indices, _ = extract_skill_codes_and_indices(
                model,
                action_np,
                device,
            )

            force_mag, torque_mag = get_skill_force_torque_mag_separate(
                trimmed,
                valid_T=valid_T,
                downsample_factor=DOWNSAMPLE_FACTOR,
            )

            H = min(len(indices), len(force_mag), len(torque_mag))

            all_skill_indices.append(np.asarray(indices[:H]))
            all_skill_force_mag.append(force_mag[:H])
            all_skill_torque_mag.append(torque_mag[:H])

            valid_eps += 1

    if len(all_skill_indices) == 0:
        print("No valid skill data found.")
        return None

    all_skill_indices = np.concatenate(all_skill_indices, axis=0)
    all_skill_force_mag = np.concatenate(all_skill_force_mag, axis=0)
    all_skill_torque_mag = np.concatenate(all_skill_torque_mag, axis=0)

    print(f"Total episodes: {total_eps}")
    print(f"Valid episodes: {valid_eps}")
    print(f"Total skill tokens: {len(all_skill_indices)}")
    print(f"Total unique skills: {len(set(all_skill_indices.tolist()))}")

    force_counts, force_token_ratios, force_bin_labels, force_edges, x_label = (
        count_unique_skills_by_bins(
            all_skill_indices,
            all_skill_force_mag,
            num_bins=num_bins,
            bin_mode=bin_mode,
        )
    )

    torque_counts, torque_token_ratios, torque_bin_labels, torque_edges, _ = (
        count_unique_skills_by_bins(
            all_skill_indices,
            all_skill_torque_mag,
            num_bins=num_bins,
            bin_mode=bin_mode,
        )
    )

    task_name = Path(task_path).name
    title_prefix = task_name if model_label is None else f"{task_name} | {model_label}"

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)
    width = 0.4

    # -----------------------------
    # Force subplot
    # -----------------------------
    x = np.arange(len(force_bin_labels))
    ax1 = axes[0]
    ax2 = ax1.twinx()

    ax1.bar(
        x - width / 2,
        force_counts,
        width=width,
        label="Unique skill count",
    )
    ax2.bar(
        x + width / 2,
        force_token_ratios,
        width=width,
        alpha=0.45,
        label="Token ratio (%)",
    )

    ax1.set_title(f"{title_prefix} - Force Magnitude ({bin_mode})")
    ax1.set_ylabel("Unique skill count")
    ax2.set_ylabel("Token ratio (%)")
    ax1.grid(axis="y", alpha=0.3)
    ax1.set_xticks(x)
    ax1.set_xticklabels(force_bin_labels, rotation=45 if bin_mode == "value" else 0, ha="right" if bin_mode == "value" else "center")
    ax1.set_xlabel(x_label)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

    # -----------------------------
    # Torque subplot
    # -----------------------------
    x = np.arange(len(torque_bin_labels))
    ax1 = axes[1]
    ax2 = ax1.twinx()

    ax1.bar(
        x - width / 2,
        torque_counts,
        width=width,
        label="Unique skill count",
    )
    ax2.bar(
        x + width / 2,
        torque_token_ratios,
        width=width,
        alpha=0.45,
        label="Token ratio (%)",
    )

    ax1.set_title(f"{title_prefix} - Torque Magnitude ({bin_mode})")
    ax1.set_ylabel("Unique skill count")
    ax2.set_ylabel("Token ratio (%)")
    ax1.set_xlabel(x_label)
    ax1.set_xticks(x)
    ax1.set_xticklabels(torque_bin_labels, rotation=45 if bin_mode == "value" else 0, ha="right" if bin_mode == "value" else "center")
    ax1.grid(axis="y", alpha=0.3)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

    plt.tight_layout()

    save_path = None
    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        safe_task = re.sub(r"[^A-Za-z0-9_.-]+", "_", task_name).strip("_")
        safe_label = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_label or "model").strip("_")
        save_path = output_dir / f"{safe_task}__{safe_label}__{bin_mode}_{num_bins}bins.png"
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved figure: {save_path}")

    plt.show()

    return {
        "skill_indices": all_skill_indices,
        "force_mag": all_skill_force_mag,
        "torque_mag": all_skill_torque_mag,
        "force_counts": force_counts,
        "force_token_ratios": force_token_ratios,
        "force_bin_labels": force_bin_labels,
        "force_edges": force_edges,
        "torque_counts": torque_counts,
        "torque_token_ratios": torque_token_ratios,
        "torque_bin_labels": torque_bin_labels,
        "torque_edges": torque_edges,
        "bin_mode": bin_mode,
        "num_bins": num_bins,
        "task_path": task_path,
        "model_label": model_label,
        "save_path": str(save_path) if save_path is not None else None,
    }


: 

In [ ]:
# Plot force/torque unique-skill histograms for every loaded autoencoder.
# Choose one task folder and one binning mode.
TASK_PATH = os.path.join(data_prefix, "multi_usb")
BIN_MODE = "percentile"  # "value" or "percentile"
NUM_BINS = 10
OUTPUT_DIR = Path.cwd() / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_autoencoder_plot_results = {}
for key, item in autoencoder_models.items():
    print("\n" + "=" * 100)
    print(key)
    print("=" * 100)
    result = plot_unique_skill_hist_force_torque_for_task(
        model=item["model"],
        task_path=TASK_PATH,
        device=device,
        num_bins=NUM_BINS,
        bin_mode=BIN_MODE,
        model_label=key,
        output_dir=OUTPUT_DIR,
    )
    all_autoencoder_plot_results[key] = result

print(f"Saved plots under: {OUTPUT_DIR}")
